<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week8_Day3_Daily_Challenge_MCP_Weather_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: MCP Weather (Ping-Pong)

Ce notebook contient la solution complète pour créer un serveur et un client MCP (Model Context Protocol) minimal.

In [1]:
# 1. Setup
!pip install -qU "mcp[cli]"
print("Installation terminée.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 4.3 MB/s eta 0:00:00
Installation terminée.


In [2]:
%%writefile server.py
import logging
import sys
from mcp.server.fastmcp import FastMCP

# Configuration du logging vers stderr pour ne pas perturber STDIO
logging.basicConfig(level=logging.INFO, stream=sys.stderr)

mcp = FastMCP("WeatherDemo")

CITY_DATA = {
    "paris": {"temp_c": 21, "condition": "ensoleillé"},
    "london": {"temp_c": 18, "condition": "nuageux"},
    "new york": {"temp_c": 24, "condition": "venteux"},
}

@mcp.tool()
def get_weather(city: str) -> dict:
    """Retourne les informations météo pour une ville supportée."""
    key = city.strip().lower()
    data = CITY_DATA.get(key)
    if not data:
        return {"error": f"Ville non supportée : {city}. Essayez : {', '.join(CITY_DATA.keys())}"}
    logging.info(f"Appel de get_weather pour {key}")
    return {"city": key, **data}

@mcp.resource("cities://list")
def list_cities() -> str:
    """Retourne la liste des villes supportées."""
    return "\n".join(sorted(CITY_DATA.keys()))

if __name__ == "__main__":
    mcp.run()

Writing server.py


In [3]:
%%writefile client.py
import asyncio
import sys
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Paramètres pour lancer le serveur via l'interpréteur actuel
SERVER_PATH = Path("server.py").absolute()
server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVER_PATH)],
    env=None
)

def extract_text(payload):
    """Aide à l'extraction du texte selon la structure de la réponse MCP"""
    if hasattr(payload, 'content'): # Pour les ressources
        return payload.content
    if hasattr(payload, 'content'): # Pour les outils (selon SDK)
        return payload.content
    return str(payload)

async def run():
    print("--- Lancement du Client MCP ---\n")
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Initialisation
            await session.initialize()

            # 1. Liste des ressources
            resources = await session.list_resources()
            print("Ressources disponibles :")
            for res in resources.resources:
                print(f"  - {res.uri} ({res.name})")

            # 2. Liste des outils
            tools = await session.list_tools()
            print("\nOutils disponibles :")
            for tool in tools.tools:
                print(f"  - {tool.name}")

            # 3. Lecture de la ressource cities://list
            print("\nLecture de cities://list :")
            cities_res = await session.read_resource("cities://list")
            print(extract_text(cities_res))

            # 4. Appel de l'outil get_weather
            print("\nAppel de get_weather('Paris') :")
            weather_result = await session.call_tool("get_weather", {"city": "Paris"})
            print(weather_result.content[0].text)

if __name__ == "__main__":
    asyncio.run(run())

Writing client.py


In [4]:
# Exécution du client (qui lance lui-même le serveur)
!python client.py

--- Lancement du Client MCP ---

INFO:mcp.server.lowlevel.server:Processing request of type ListResourcesRequest
Ressources disponibles :
  - cities://list (list_cities)
INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest

Outils disponibles :
  - get_weather

Lecture de cities://list :
INFO:mcp.server.lowlevel.server:Processing request of type ReadResourceRequest
meta=None contents=[TextResourceContents(uri=AnyUrl('cities://list'), mimeType='text/plain', meta=None, text='london\nnew york\nparis')]

Appel de get_weather('Paris') :
INFO:mcp.server.lowlevel.server:Processing request of type CallToolRequest
INFO:root:Appel de get_weather pour paris
{
  "city": "paris",
  "temp_c": 21,
  "condition": "ensoleillé"
}


# Daily Challenge: MCP Weather (Student)
Beginner-friendly MCP server + client (no LLMs). Complete the TODOs, then run the client.

## Setup
Run the install cell. If Colab asks, restart runtime after install.

In [5]:
# Install MCP CLI + SDK
%pip install -qU "mcp[cli]"

In [6]:
# Quick verify
!python --version
!mcp --help | head -n 5

Python 3.12.13
                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 MCP development tools                                                          
                                                                                


## A) Server (server.py)
Implement the WeatherDemo server with one tool and one resource.

In [7]:

%%writefile server.py
import logging
from mcp.server.fastmcp import FastMCP

logging.basicConfig(level=logging.INFO)

mcp = FastMCP("WeatherDemo")

CITY_DATA = {
    "paris": {"temp_c": 21, "condition": "sunny"},
    "london": {"temp_c": 18, "condition": "cloudy"},
    "new york": {"temp_c": 24, "condition": "breezy"},
}

@mcp.tool()
def get_weather(city: str) -> dict:
    """Return simple weather info for a supported city."""
    key = city.strip().lower()
    data = CITY_DATA.get(key)
    if not data:
        return {"error": f"Unsupported city: {city}. Try one of: {', '.join(CITY_DATA)}"}
    logging.info("get_weather called for %s", key)
    return {"city": key, **data}

@mcp.resource("cities://list")
def list_cities() -> str:
    """Return supported cities as newline-separated text."""
    return "".join(sorted(CITY_DATA))

if __name__ == "__main__":
    mcp.run()


Overwriting server.py


## B) Client (client.py)
Spawn the server via STDIO, discover capabilities, and call them.

In [8]:

%%writefile client.py
import asyncio
import sys
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Use local server.py with this interpreter to avoid PATH conflicts
SERVER_PATH = Path(__file__).parent / "server.py"
server_params = StdioServerParameters(
    command=sys.executable, args=[str(SERVER_PATH)], env=None
)

def extract_content(payload):
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        return payload.content
    return str(payload)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            resources = await session.list_resources()
            print("Resources:")
            for res in resources.resources:
                print(f"- {res.uri} ({res.name or ''})")

            tools = await session.list_tools()
            print("Tools:")
            for tool in tools.tools:
                print(f"- {tool.name}")

            cities = await session.read_resource("cities://list")
            print("cities://list ->")
            print(extract_content(cities))

            weather = await session.call_tool("get_weather", {"city": "Paris"})
            print("get_weather(Paris) ->", extract_content(weather))


if __name__ == "__main__":
    asyncio.run(run())


Overwriting client.py


## C) Run
Single terminal (spawns server):
```
python client.py
```
Or two terminals for debugging:
```
mcp run server.py
python client.py
```
In Colab, just run the next cell.

In [ ]:
!python client.py

## Troubleshooting
- `mcp: command not found` ? rerun install or restart runtime.
- No tools/resources ? check decorators, restart server.
- City missing ? use one of the supported cities listed by `cities://list`.